## POC de la creacion dinamica de los sets de entrenamiento (train/test)


- 1) Depende de el entrenamiento que queramos hacer
- 2) Seleccionar features
- 3) Calcular nuevas features (de forma dinamica - dado x hiperparámetros)
- 4) Limpiar registros (null, incoherencias...)
- 5) Definir train-test (de forma dinamica)
- 6) Definir split random/squential (de forma dinamica)
- 7) ¿Como integrar esto en el workflow actual?

## MIS MODELOS

In [1]:

import pandas as pd

In [2]:
cols_metadata = ['matchId','id_H', 'id_A','Div','div_order', 'Date','season','HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR','Attendance','Referee']
cols_features = ['HS', 'AS', 'HST', 'AST', 'HF','AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR','HO','AO','HHW','AHW']
cols_bets     = ['B365H', 'B365D', 'B365A','MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']

In [3]:
data = pd.read_csv('F:\\TFG\\datasets\\raw_datasets\\datalake.csv',sep=';',decimal=',',parse_dates=['Date'],date_format="%d/%m/%Y")
data

<ipython-input-3-802293c8dba1>:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('F:\\TFG\\datasets\\raw_datasets\\datalake.csv',sep=';',decimal=',',parse_dates=['Date'],date_format="%d/%m/%Y")


,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,D100000,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D100001,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D100002,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D100003,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D100004,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,SP210828,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58542,SP210829,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58543,SP210830,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58544,SP210831,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [133]:
(~data.FTHG.isna()).mean()  

1.0

In [134]:
# util functions

def getPoints(scored,received):
    if scored>received: return 3
    if scored==received: return 1
    else: return 0

In [135]:
data

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,D100000,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D100001,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D100002,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D100003,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D100004,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,SP210828,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58542,SP210829,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58543,SP210830,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58544,SP210831,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [136]:
cols_H = ['matchId','Div','Date','id_H','HomeTeam','FTHG','FTAG','FTR','HS','HST','HF','HC', 'HY','HR','HO','HHW']
cols_A = ['matchId','Div','Date','id_A','AwayTeam','FTAG','FTHG','FTR','AS','AST','AF','AC', 'AY','AR','AO','AHW']
cols_aux = ['matchId','Div','Date','idTeam','Team','FTG','FTG_rival','FTR','S','ST','F','C', 'Y','R','O','HW']

df_H = data[cols_H]
df_H.columns = cols_aux
df_H['Side'] = 0
df_A = data[cols_A]
df_A.columns = cols_aux
df_A['Side'] = 1

<ipython-input-136-11223f2d7926>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_H['Side'] = 0
<ipython-input-136-11223f2d7926>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_A['Side'] = 1


In [137]:
df_A

,matchId,Div,Date,idTeam,Team,FTG,FTG_rival,FTR,S,ST,F,C,Y,R,O,HW,Side
0,D100000,D1,2000-11-08,19,Hansa Rostock,0,1,H,5,2,19,3,5,0,8.0,0.0,1
1,D100001,D1,2000-12-08,20,Hertha,1,4,H,11,5,12,9,0,0,3.0,0.0,1
2,D100002,D1,2000-12-08,35,Stuttgart,0,4,H,18,5,17,7,1,0,0.0,0.0,1
3,D100003,D1,2000-12-08,29,Munich 1860,2,2,D,9,7,0,3,2,1,0.0,0.0,1
4,D100004,D1,2000-12-08,4,Bochum,1,0,A,5,2,8,5,0,0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,SP210828,SP2,2000-04-06,243,Badajoz,2.0,2.0,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
58542,SP210829,SP2,2000-04-06,274,Levante,1.0,2.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
58543,SP210830,SP2,2000-04-06,314,Villarreal,0.0,1.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
58544,SP210831,SP2,2000-04-06,297,Recreativo,1.0,2.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [138]:
df = pd.concat([df_H,df_A])
df.sort_values(['Date','matchId'],ascending=True,inplace=True)
df

,matchId,Div,Date,idTeam,Team,FTG,FTG_rival,FTR,S,ST,F,C,Y,R,O,HW,Side
6316,D106316,D1,1993-01-09,2,Bayern Munich,3.0,0.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6316,D106316,D1,1993-01-09,25,Leipzig,0.0,3.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
6317,D106317,D1,1993-01-09,8,Dortmund,4.0,0.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6317,D106317,D1,1993-01-09,9,Dresden,0.0,4.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
6318,D106318,D1,1993-01-09,12,Ein Frankfurt,3.0,1.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44863,SP107842,SP1,2021-12-05,200,Getafe,0,1,H,13,4,18,5,3,0,NaN,NaN,1
44864,SP107843,SP1,2021-12-05,205,Huesca,1,0,H,18,6,9,5,0,0,NaN,NaN,0
44864,SP107843,SP1,2021-12-05,188,Ath Bilbao,0,1,H,4,3,14,0,1,0,NaN,NaN,1
44865,SP107844,SP1,2021-12-05,189,Ath Madrid,2,1,H,12,4,11,1,0,0,NaN,NaN,0


In [139]:
npj = '30D'
cols_to_group = ['idTeam']
aggregations  = {
    "FTG":"mean","FTG_rival":"mean",
    "S":"mean", "ST":"mean",
    "F":"mean", "C":"mean",
    "Y":"mean", "R":"mean",
    "O":"mean", "HW":"mean", 
    "Side":"mean"
}

cols_agg = [ k+"_"+npj for k in aggregations.keys() ]

df_agg = (df.groupby(cols_to_group)
    .rolling(window=npj,min_periods=2,on='Date',closed='left')
    .agg(aggregations)
)

df_agg.columns = cols_agg
df_agg

FTG_30D  FTG_rival_30D  S_30D  ST_30D  F_30D  C_30D  Y_30D   
idTeam Date                                                                     
0      2006-03-12      NaN            NaN    NaN     NaN    NaN    NaN    NaN  \
       2006-04-11      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2006-07-11      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2006-08-19      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2006-08-26      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
...                    ...            ...    ...     ...    ...    ...    ...   
317    2021-08-01      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2021-08-05      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2021-11-04      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2021-12-02      NaN            NaN    NaN     NaN    NaN    NaN    NaN   
       2021-12-03      1.5            1.0   11.0     4.5   14.0    3.0    1.5   

                   R_30D  O_30D  HW_30D  Side_30D  
idTeam Date                                        
0      2006-03-12    NaN    NaN     NaN       NaN  
       2006-04-11    NaN    NaN     NaN       NaN  
       2006-07-11    NaN    NaN     NaN       NaN  
       2006-08-19    NaN    NaN     NaN       NaN  
       2006-08-26    NaN    NaN     NaN       NaN  
...                  ...    ...     ...       ...  
317    2021-08-01    NaN    NaN     NaN       NaN  
       2021-08-05    NaN    NaN     NaN       NaN  
       2021-11-04    NaN    NaN     NaN       NaN  
       2021-12-02    NaN    NaN     NaN       NaN  
       2021-12-03    0.0    NaN     NaN       0.5  

[117092 rows x 11 columns]

In [140]:
df_count = data.groupby(['id_H','Date']).count()
df_count[df_count.matchId>1]#[df_count.Side>1]

,,matchId,id_A,Div,div_order,season,HomeTeam,AwayTeam,FTHG,FTAG,FTR,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
id_H,Date,,,,,,,,,,,,,,,,,,,,,


In [141]:
df_merged = df.merge(df_agg.reset_index(),on=["idTeam","Date"],how='left')
df_merged

,matchId,Div,Date,idTeam,Team,FTG,FTG_rival,FTR,S,ST,...,FTG_rival_30D,S_30D,ST_30D,F_30D,C_30D,Y_30D,R_30D,O_30D,HW_30D,Side_30D
0,D106316,D1,1993-01-09,2,Bayern Munich,3.0,0.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D106316,D1,1993-01-09,25,Leipzig,0.0,3.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D106317,D1,1993-01-09,8,Dortmund,4.0,0.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D106317,D1,1993-01-09,9,Dresden,0.0,4.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D106318,D1,1993-01-09,12,Ein Frankfurt,3.0,1.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,SP107842,SP1,2021-12-05,200,Getafe,0,1,H,13,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117088,SP107843,SP1,2021-12-05,205,Huesca,1,0,H,18,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117089,SP107843,SP1,2021-12-05,188,Ath Bilbao,0,1,H,4,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117090,SP107844,SP1,2021-12-05,189,Ath Madrid,2,1,H,12,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### TEST

In [3]:
import experiments.dataflow_own as own
import experiments.utils as ut
import importlib as im

In [79]:
options = {
    "experiment": "own",
    "data":{
        "aggregations": {
            "FTG":"mean","FTG_rival":"mean",
            "S":"mean", "ST":"mean",
            "F":"mean", "C":"mean",
            "Y":"mean", "R":"mean",
            "O":"mean", "HW":"mean", 
            "Side":"mean"
        },
        "features": [],
        "leagues": [],
        "lags": ["15D","30D","730D",],
        "min_samples": [1,2,20],
        "col_group": ["idTeam"],
        "col_window": "Date",
        "split": 0.75,
        "sample": "sequential",
        "split_date":"2018-01-01",
    },
    "paths":{
    },
    "iterations":1000,

}

In [82]:
im.reload(ut)
im.reload(own)
rawdata = ut.read_data("F:\\TFG\\datasets\\raw_datasets\\datalake.csv")
Dataset = own.Dataflow_own(rawdata,options['data'],options['paths'])
# data = own.run(rawdata,options['data'],options['paths'])

ModuleNotFoundError: No module named 'utils'

In [43]:
Dataset.features

array(['FTG_15D_H', 'FTG_15D_A', 'FTG_30D_H', 'FTG_30D_A', 'FTG_730D_H',
       'FTG_730D_A', 'FTG_rival_15D_H', 'FTG_rival_15D_A',
       'FTG_rival_30D_H', 'FTG_rival_30D_A', 'FTG_rival_730D_H',
       'FTG_rival_730D_A', 'S_15D_H', 'S_15D_A', 'S_30D_H', 'S_30D_A',
       'S_730D_H', 'S_730D_A', 'ST_15D_H', 'ST_15D_A', 'ST_30D_H',
       'ST_30D_A', 'ST_730D_H', 'ST_730D_A', 'F_15D_H', 'F_15D_A',
       'F_30D_H', 'F_30D_A', 'F_730D_H', 'F_730D_A', 'C_15D_H', 'C_15D_A',
       'C_30D_H', 'C_30D_A', 'C_730D_H', 'C_730D_A', 'Y_15D_H', 'Y_15D_A',
       'Y_30D_H', 'Y_30D_A', 'Y_730D_H', 'Y_730D_A', 'R_15D_H', 'R_15D_A',
       'R_30D_H', 'R_30D_A', 'R_730D_H', 'R_730D_A', 'O_15D_H', 'O_15D_A',
       'O_30D_H', 'O_30D_A', 'O_730D_H', 'O_730D_A', 'HW_15D_H',
       'HW_15D_A', 'HW_30D_H', 'HW_30D_A', 'HW_730D_H', 'HW_730D_A',
       'Side_15D_H', 'Side_15D_A', 'Side_30D_H', 'Side_30D_A',
       'Side_730D_H', 'Side_730D_A'], dtype='<U16')

In [44]:
data = Dataset.df

In [45]:
data

,matchId,Div,Date,season,idTeam_H,Team_H,FTG_H,FTG_rival_H,FTR_H,S_H,...,FTG_rival_730D_A,S_730D_A,ST_730D_A,F_730D_A,C_730D_A,Y_730D_A,R_730D_A,O_730D_A,HW_730D_A,Side_730D_A
0,D106316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D106317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D106318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D106319,D1,1993-01-09,T93-94,13,FC Koln,2.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D106320,D1,1993-01-09,T93-94,17,Hamburg,2.0,1.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,I107469,I1,2021-12-05,T20-21,179,Torino,0,7,A,14,...,1.172414,12.551724,6.810345,13.758621,5.758621,2.224138,0.086207,NaN,NaN,0.482759
58542,SP107841,SP1,2021-12-05,T20-21,223,Sevilla,1,0,H,10,...,1.389831,9.576271,3.338983,11.644068,4.288136,2.152542,0.169492,NaN,NaN,0.491525
58543,SP107842,SP1,2021-12-05,T20-21,193,Celta,1,0,H,4,...,1.067797,10.050847,3.016949,17.322034,4.186441,3.389831,0.186441,NaN,NaN,0.491525
58544,SP107843,SP1,2021-12-05,T20-21,205,Huesca,1,0,H,18,...,1.135593,10.728814,3.474576,13.033898,5.050847,2.305085,0.084746,NaN,NaN,0.491525


In [109]:
print("Goles: ",sum(~data.FTG_10D.isna()) * 100 / len(data))
print("Goles: ",sum(~data.FTG_15D.isna()) * 100 / len(data))
print("Goles: ",sum(~data.FTG_120D.isna()) * 100 / len(data))
print("Goles: ",sum(~data.FTG_365D.isna()) * 100 / len(data))
print("Goles: ",sum(~data.FTG_730D.isna()) * 100 / len(data))
print("Goles: ",sum(~data.FTG_1825D.isna()) * 100 / len(data))
print("Disparos al palo: ", sum(~data.HW_30D.isna()) * 100 / len(data))
print("Rojas: ", sum(~data.R_10D.isna()) * 100 / len(data))
print("Rojas: ", sum(~data.R_120D.isna()) * 100 / len(data))
print("Rojas: ", sum(~data.R_1825D.isna()) * 100 / len(data))

AttributeError: 'DataFrame' object has no attribute 'FTG_10D'

In [128]:
mask_home = data.Side==0
mask_away = data.Side==1

data_merged = data[mask_home].merge(data[mask_away], on=['matchId','Div','Date'], suffixes=("_H","_A"))
data_merged

,matchId,Div,Date,idTeam_H,Team_H,FTG_H,FTG_rival_H,FTR_H,S_H,ST_H,...,FTG_rival_730D_A,S_730D_A,ST_730D_A,F_730D_A,C_730D_A,Y_730D_A,R_730D_A,O_730D_A,HW_730D_A,Side_730D_A
0,D106316,D1,1993-01-09,2,Bayern Munich,3.0,0.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D106317,D1,1993-01-09,8,Dortmund,4.0,0.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D106318,D1,1993-01-09,12,Ein Frankfurt,3.0,1.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D106319,D1,1993-01-09,13,FC Koln,2.0,0.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D106320,D1,1993-01-09,17,Hamburg,2.0,1.0,H,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,I107469,I1,2021-12-05,179,Torino,0,7,A,14,1,...,1.172414,12.551724,6.810345,13.758621,5.758621,2.224138,0.086207,NaN,NaN,0.482759
58542,SP107841,SP1,2021-12-05,223,Sevilla,1,0,H,10,4,...,1.389831,9.576271,3.338983,11.644068,4.288136,2.152542,0.169492,NaN,NaN,0.491525
58543,SP107842,SP1,2021-12-05,193,Celta,1,0,H,4,1,...,1.067797,10.050847,3.016949,17.322034,4.186441,3.389831,0.186441,NaN,NaN,0.491525
58544,SP107843,SP1,2021-12-05,205,Huesca,1,0,H,18,6,...,1.135593,10.728814,3.474576,13.033898,5.050847,2.305085,0.084746,NaN,NaN,0.491525


In [46]:
data.columns

Index(['matchId', 'Div', 'Date', 'season', 'idTeam_H', 'Team_H', 'FTG_H',
       'FTG_rival_H', 'FTR_H', 'S_H', 'ST_H', 'F_H', 'C_H', 'Y_H', 'R_H',
       'O_H', 'HW_H', 'Side_H', 'FTG_15D_H', 'FTG_rival_15D_H', 'S_15D_H',
       'ST_15D_H', 'F_15D_H', 'C_15D_H', 'Y_15D_H', 'R_15D_H', 'O_15D_H',
       'HW_15D_H', 'Side_15D_H', 'FTG_30D_H', 'FTG_rival_30D_H', 'S_30D_H',
       'ST_30D_H', 'F_30D_H', 'C_30D_H', 'Y_30D_H', 'R_30D_H', 'O_30D_H',
       'HW_30D_H', 'Side_30D_H', 'FTG_730D_H', 'FTG_rival_730D_H', 'S_730D_H',
       'ST_730D_H', 'F_730D_H', 'C_730D_H', 'Y_730D_H', 'R_730D_H', 'O_730D_H',
       'HW_730D_H', 'Side_730D_H', 'idTeam_A', 'Team_A', 'FTG_A',
       'FTG_rival_A', 'FTR_A', 'S_A', 'ST_A', 'F_A', 'C_A', 'Y_A', 'R_A',
       'O_A', 'HW_A', 'Side_A', 'FTG_15D_A', 'FTG_rival_15D_A', 'S_15D_A',
       'ST_15D_A', 'F_15D_A', 'C_15D_A', 'Y_15D_A', 'R_15D_A', 'O_15D_A',
       'HW_15D_A', 'Side_15D_A', 'FTG_30D_A', 'FTG_rival_30D_A', 'S_30D_A',
       'ST_30D_A', 'F_30D_A',

In [47]:
last_digit = lambda x: x[-1]
data['last_digit'] = data.matchId.apply(last_digit).astype(int)
data

,matchId,Div,Date,season,idTeam_H,Team_H,FTG_H,FTG_rival_H,FTR_H,S_H,...,S_730D_A,ST_730D_A,F_730D_A,C_730D_A,Y_730D_A,R_730D_A,O_730D_A,HW_730D_A,Side_730D_A,last_digit
0,D106316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
1,D106317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
2,D106318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
3,D106319,D1,1993-01-09,T93-94,13,FC Koln,2.0,0.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
4,D106320,D1,1993-01-09,T93-94,17,Hamburg,2.0,1.0,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,I107469,I1,2021-12-05,T20-21,179,Torino,0,7,A,14,...,12.551724,6.810345,13.758621,5.758621,2.224138,0.086207,NaN,NaN,0.482759,9
58542,SP107841,SP1,2021-12-05,T20-21,223,Sevilla,1,0,H,10,...,9.576271,3.338983,11.644068,4.288136,2.152542,0.169492,NaN,NaN,0.491525,1
58543,SP107842,SP1,2021-12-05,T20-21,193,Celta,1,0,H,4,...,10.050847,3.016949,17.322034,4.186441,3.389831,0.186441,NaN,NaN,0.491525,2
58544,SP107843,SP1,2021-12-05,T20-21,205,Huesca,1,0,H,18,...,10.728814,3.474576,13.033898,5.050847,2.305085,0.084746,NaN,NaN,0.491525,3


In [48]:
test = [5,9]
mask_test = (data.last_digit==5) | (data.last_digit==9)
data_train = data[~mask_test]
data_test = data[mask_test]

data_train.shape, data_test.shape

((46840, 99), (11706, 99))

In [49]:
COLS_META = ['matchId','Div','Date','season']

In [75]:
data_train = data[data.Date<"2018-01-01"]
data_test  = data[data.Date>"2018-01-01"]

In [76]:
len(data_test)/len(data)

0.1332968947494278

In [77]:
data_test.groupby(["Div","season"]).matchId.count().unstack(0)

Div,D1,E0,F1,I1,SP1,SP2
season,,,,,,
T17-18,153,166,190,192,211,243
T18-19,306,380,380,380,380,462
T19-20,306,380,279,380,380,462
T20-21,288,350,380,358,380,418


In [78]:
data_train.groupby(["Div","season"]).matchId.count().unstack(0)

Div,D1,E0,F1,I1,SP1,SP2
season,,,,,,
T00-01,306.0,380.0,306.0,306.0,380.0,462.0
T01-02,306.0,380.0,306.0,306.0,380.0,414.0
T02-03,290.0,380.0,191.0,306.0,380.0,239.0
T03-04,194.0,335.0,223.0,194.0,380.0,158.0
T04-05,306.0,380.0,380.0,300.0,268.0,462.0
T05-06,306.0,335.0,380.0,380.0,380.0,462.0
T06-07,306.0,380.0,380.0,380.0,380.0,462.0
T07-08,306.0,380.0,380.0,380.0,380.0,446.0
T08-09,306.0,380.0,380.0,380.0,380.0,462.0
